# Year Integration

**Goal:** align the 2025 and 2026 Oracle's Elixir extracts, rebuild a unified match-level dataset, engineer pre-match features, and confirm whether the combined data supports a stronger baseline model.

This notebook answers:

- do 2025 and 2026 expose a compatible core schema?
- can the yearly tables be standardized cleanly?
- what date range and match counts are covered after integration?
- are season labels consistent with the parsed match dates?
- what feature dataset should be exported for downstream modeling?

## 1) Load Raw Yearly Extracts

Read the two raw Oracle's Elixir exports that will be standardized and merged into a single season-spanning dataset.

In [1]:
# Import pandas for dataframe loading and column-level inspection.
import pandas as pd

# Load the raw yearly extracts that will be reconciled into one pipeline.
df_2026 = pd.read_csv("../data/raw/2026_LoL_OraclesElixir.csv", low_memory=False)
df_2025 = pd.read_csv("../data/raw/2025_LoL_OraclesElixir.csv", low_memory=False)

## 2) Compare Schemas and Standardize Shared Columns

Confirm that both yearly files share the fields needed for integration, then restrict the data to comparable team-level rows.

In [2]:
# Identify the intersection of yearly schemas before selecting a common subset.
common_cols = sorted(set(df_2025.columns).intersection(df_2026.columns))
len(common_cols), common_cols[:]

(165,
 ['assists',
  'assistsat10',
  'assistsat15',
  'assistsat20',
  'assistsat25',
  'atakhans',
  'ban1',
  'ban2',
  'ban3',
  'ban4',
  'ban5',
  'barons',
  'champion',
  'chemtechs',
  'ckpm',
  'clouds',
  'controlwardsbought',
  'csat10',
  'csat15',
  'csat20',
  'csat25',
  'csdiffat10',
  'csdiffat15',
  'csdiffat20',
  'csdiffat25',
  'cspm',
  'damagemitigatedperminute',
  'damageshare',
  'damagetakenperminute',
  'damagetochampions',
  'damagetotowers',
  'datacompleteness',
  'date',
  'deaths',
  'deathsat10',
  'deathsat15',
  'deathsat20',
  'deathsat25',
  'doublekills',
  'dpm',
  'dragons',
  'dragons (type unknown)',
  'earned gpm',
  'earnedgold',
  'earnedgoldshare',
  'elders',
  'elementaldrakes',
  'firstPick',
  'firstbaron',
  'firstblood',
  'firstbloodassist',
  'firstbloodkill',
  'firstbloodvictim',
  'firstdragon',
  'firstherald',
  'firstmidtower',
  'firsttothreetowers',
  'firsttower',
  'game',
  'gameid',
  'gamelength',
  'goldat10',
  'gold

In [3]:
# Keep only the core columns required to rebuild match-level rows consistently across seasons.
core_cols = [
    "gameid",
    "date",
    "patch",
    "league",
    "side",
    "teamname",
    "result",
    "position",
    "datacompleteness",
]

# Work on narrowed copies so downstream transforms only touch the shared schema.
df_2025 = df_2025[core_cols].copy()
df_2026 = df_2026[core_cols].copy()

# Retain only complete team-level records for leak-free match reconstruction.
df_2025 = df_2025[df_2025["datacompleteness"] == "complete"]
df_2026 = df_2026[df_2026["datacompleteness"] == "complete"]

df_2025 = df_2025[df_2025["position"] == "team"]
df_2026 = df_2026[df_2026["position"] == "team"]

# Normalize side labels and stamp the source season on each row.
df_2025["side"] = df_2025["side"].str.lower()
df_2026["side"] = df_2026["side"].str.lower()

df_2025["year"] = 2025
df_2026["year"] = 2026

# Preview the cleaned 2025 frame after filtering and standardization.
df_2025.head()

,gameid,date,patch,league,side,teamname,result,position,datacompleteness,year
10,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,blue,IziDream,0,team,complete,2025
11,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,red,Team Valiant,1,team,complete,2025
22,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,blue,Esprit Shōnen,1,team,complete,2025
23,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,red,Skillcamp,0,team,complete,2025
34,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,blue,Karmine Corp Blue Stars,0,team,complete,2025


In [4]:
# Count retained 2025 team rows after filtering to the comparable subset.
df_2025.count()

gameid              18472
date                18472
patch               18472
league              18472
side                18472
teamname            18472
result              18472
position            18472
datacompleteness    18472
year                18472
dtype: int64

In [5]:
# Preview the cleaned 2026 frame to confirm the same structure was applied.
df_2026.head()

,gameid,date,patch,league,side,teamname,result,position,datacompleteness,year
10,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,blue,GMBLERS Esports,0,team,complete,2026
11,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,red,EKO Esports,1,team,complete,2026
22,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,blue,Deacoy,1,team,complete,2026
23,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,red,Zena Esports,0,team,complete,2026
34,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,blue,Axolotl,0,team,complete,2026


In [6]:
# Count retained 2026 team rows after filtering to the comparable subset.
df_2026.count()

gameid              8572
date                8572
patch               8572
league              8572
side                8572
teamname            8572
result              8572
position            8572
datacompleteness    8572
year                8572
dtype: int64

## 3) Combine Both Seasons and Rebuild Match Rows

Append the standardized team-level rows, sort them chronologically, and collapse each two-row game into a single match-level record.

In [7]:
# Stack the cleaned yearly tables into one chronological raw team-level dataset.
combined_raw = pd.concat([df_2025, df_2026], ignore_index=True)
combined_raw["date"] = pd.to_datetime(combined_raw["date"])
combined_raw = combined_raw.sort_values(by="date").reset_index(drop=True)

# Count non-null values after the merge to spot obvious integration issues.
combined_raw.count()

gameid              27044
date                27044
patch               27044
league              27044
side                27044
teamname            27044
result              27044
position            27044
datacompleteness    27044
year                27044
dtype: int64

In [8]:
# Keep the columns required to collapse two team rows into one match row.
core_cols = [
    "gameid",
    "date",
    "patch",
    "league",
    "year",
    "side",
    "teamname",
    "result",
]

team_df = combined_raw[core_cols].copy()

# Collect one match-level record per valid game id.
matches = []

for gameid, group in team_df.groupby("gameid"):
    # Only keep games with exactly one blue-side and one red-side team row.
    if len(group) != 2:
        continue

    blue = group[group["side"] == "blue"]
    red = group[group["side"] == "red"]
    
    if blue.empty or red.empty:
        continue

    blue_row = blue.iloc[0]
    red_row = red.iloc[0]
    
    # Rebuild a single pre-match row keyed by game id.
    matches.append({
        "gameid": gameid,
        "date": blue_row["date"],
        "patch": blue_row["patch"],
        "league": blue_row["league"],
        "year": blue_row["year"],
        "blue_team": blue_row["teamname"],
        "red_team": red_row["teamname"],
        "blue_side_win": int(blue_row["result"]),
    })

combined_matches_df = pd.DataFrame(matches)
combined_matches_df.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win
0,LOLTMNT01_189225,2025-01-18 14:32:40,15.01,LCP,2025,Chiefs Esports Club,PSG Talon,0
1,LOLTMNT01_189281,2025-01-19 11:42:16,15.01,LCP,2025,MGN Vikings Esports,GAM Esports,1
2,LOLTMNT01_189290,2025-01-19 13:25:52,15.01,LCP,2025,Fukuoka SoftBank HAWKS gaming,CTBC Flying Oyster,0
3,LOLTMNT01_189291,2025-01-19 14:29:13,15.01,LCP,2025,Fukuoka SoftBank HAWKS gaming,CTBC Flying Oyster,0
4,LOLTMNT01_189329,2025-01-21 14:10:53,15.01,HLL,2025,Gamespace Mediterranean College Esports,Team Refuse,1


## 4) Validate the Integrated Match Table

Inspect the rebuilt match-level dataset to confirm row counts, null handling, label balance, and season coverage before exporting it.

In [9]:
# Inspect dataframe shape, dtypes, and non-null counts for the rebuilt match table.
combined_matches_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13522 entries, 0 to 13521
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   gameid         13522 non-null  object        
 1   date           13522 non-null  datetime64[ns]
 2   patch          13522 non-null  float64       
 3   league         13522 non-null  object        
 4   year           13522 non-null  int64         
 5   blue_team      13522 non-null  object        
 6   red_team       13522 non-null  object        
 7   blue_side_win  13522 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 845.3+ KB


In [10]:
# Verify that required match-level fields are populated after reconstruction.
combined_matches_df.isnull().sum()

gameid           0
date             0
patch            0
league           0
year             0
blue_team        0
red_team         0
blue_side_win    0
dtype: int64

In [11]:
# Count how many integrated matches come from each source season.
combined_matches_df["year"].value_counts()

year
2025    9236
2026    4286
Name: count, dtype: int64

In [12]:
# Check blue-side win balance in the combined match table.
combined_matches_df["blue_side_win"].value_counts()

blue_side_win
1    7179
0    6343
Name: count, dtype: int64

In [13]:
# Confirm the final row and column count of the rebuilt match table.
combined_matches_df.shape

(13522, 8)

## 5) Export the Integrated Match Table and Check Year Labels

Write the match-level dataset to the processed directory, then verify that the stored season label agrees with the parsed match date.

In [14]:
# Persist the rebuilt match-level table for reuse in scripts and later notebooks.
combined_matches_df.to_csv("../data/processed/combined_processed_matches.csv", index=False)

In [15]:
# Cross-check the parsed date year against the explicit source-year label.
combined_matches_df["date_year"] = combined_matches_df["date"].dt.year
combined_matches_df[combined_matches_df["date_year"] != combined_matches_df["year"]]

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win,date_year


## 6) Engineer Pre-Match Historical Features

Create leak-free all-time team form features by walking through the integrated matches in chronological order and using only prior results.

In [16]:
# Ensure matches are processed in chronological order before computing historical features.
combined_matches_df = combined_matches_df.sort_values("date").reset_index(drop=True)

# Track each team's cumulative wins and games observed before the current match.
team_totals = {}
feature_rows = []

for _, row in combined_matches_df.iterrows():
    blue = row["blue_team"]
    red = row["red_team"]

    if blue not in team_totals:
        team_totals[blue] = {"wins": 0, "games": 0}
    if red not in team_totals:
        team_totals[red] = {"wins": 0, "games": 0}

    # Compute each team's all-time pre-match form from prior observed games only.
    blue_games = team_totals[blue]["games"]
    red_games = team_totals[red]["games"]

    blue_wr = team_totals[blue]["wins"] / blue_games if blue_games > 0 else 0.5
    red_wr = team_totals[red]["wins"] / red_games if red_games > 0 else 0.5

    # Save the engineered snapshot alongside the original match-level columns.
    feature_rows.append({
        **row,
        "blue_team_wr": blue_wr,
        "red_team_wr": red_wr,
        "blue_team_games": blue_games,
        "red_team_games": red_games,
        "wr_diff": blue_wr - red_wr,
    })

    # Update team histories after observing the match outcome.
    if row["blue_side_win"] == 1:
        team_totals[blue]["wins"] += 1
    else:
        team_totals[red]["wins"] += 1

    team_totals[blue]["games"] += 1
    team_totals[red]["games"] += 1

combined_feature_df = pd.DataFrame(feature_rows)
combined_feature_df.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win,date_year,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff
0,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,2025,IziDream,Team Valiant,0,2025,0.5,0.5,0,0,0.0
1,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,2025,Esprit Shōnen,Skillcamp,1,2025,0.5,0.5,0,0,0.0
2,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,2025,Karmine Corp Blue Stars,Project Conquerors,0,2025,0.5,0.5,0,0,0.0
3,LOLTMNT03_178705,2025-01-11 14:03:27,15.01,LFL2,2025,Zerance,Lille Esport,0,2025,0.5,0.5,0,0,0.0
4,LOLTMNT06_96169,2025-01-12 11:04:21,15.01,LFL2,2025,Team Valiant,Karmine Corp Blue Stars,1,2025,1.0,0.0,1,1,1.0


## 7) Inspect and Export the Feature Dataset

Run quick diagnostics on the engineered features, confirm class and season coverage after integration, and save the final modeling table.

In [17]:
# Summarize the distribution of the engineered win-rate features.
combined_feature_df[["blue_team_wr", "red_team_wr", "wr_diff"]].describe()

,blue_team_wr,red_team_wr,wr_diff
count,13522.000000,13522.000000,13522.000000
mean,0.517726,0.522799,-0.005073
std,0.192842,0.194237,0.266991
min,0.000000,0.000000,-1.000000
25%,0.414717,0.418605,-0.146005
50%,0.530864,0.529412,0.000000
75%,0.636364,0.636364,0.142825
max,1.000000,1.000000,1.000000


In [18]:
# Summarize how much historical match volume each side contributes pre-match.
combined_feature_df[["blue_team_games", "red_team_games"]].describe()

,blue_team_games,red_team_games
count,13522.000000,13522.000000
mean,48.474190,47.443869
std,44.032508,43.547870
min,0.000000,0.000000
25%,14.000000,14.000000
50%,36.000000,35.000000
75%,71.000000,69.000000
max,244.000000,242.000000


In [19]:
# Confirm that the engineered feature table still spans both seasons.
combined_feature_df["year"].value_counts()

year
2025    9236
2026    4286
Name: count, dtype: int64

In [20]:
# Check class balance in the final feature dataset used for modeling.
combined_feature_df["blue_side_win"].value_counts()

blue_side_win
1    7179
0    6343
Name: count, dtype: int64

In [21]:
# Export the combined feature dataset for downstream model training and evaluation.
combined_feature_df.to_csv("../data/processed/combined_feature_matches.csv", index=False)